# Modelo de celdas (B2)
Aprende a reconocer que es cada celda de un horario en Excel: `dia`, `hora`, `nombre`, `prueba` (clase de prueba, marca CP) u `otro`.
Datos: los Excels inventados de `datos/generar_excels.py` (ejecutalo antes). Rasgos: `rasgos_celdas.py` (el mismo codigo que usa el lector).
Separamos por archivo: 250 Excels para aprender y 50 de examen. Objetivo: >= 98% de celdas bien en el examen.

In [1]:
import sys
from pathlib import Path
import joblib
import pandas as pd
from openpyxl import load_workbook
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
from rasgos_celdas import ETIQUETAS, rasgos_hoja

GEN = RAIZ / "datos" / "generados"
etiquetas = pd.read_csv(GEN / "etiquetas.csv")
etiquetas.etiqueta.value_counts()

etiqueta
nombre    166801
hora       35739
otro       24151
dia        16452
prueba     13192
Name: count, dtype: int64

In [2]:
filas = []
for archivo in sorted(etiquetas.archivo.unique()):
    wb = load_workbook(GEN / "excels" / archivo, data_only=True)
    for ws in wb.worksheets:
        for f, c, _, r in rasgos_hoja(ws):
            filas.append({"archivo": archivo, "hoja": ws.title, "fila": f, "col": c, **r})
X = pd.DataFrame(filas).merge(etiquetas[["archivo", "hoja", "fila", "col", "etiqueta"]],
                              on=["archivo", "hoja", "fila", "col"], how="left", validate="one_to_one")
assert X.etiqueta.notna().all(), "Hay celdas sin etiqueta"
COLS = [c for c in X.columns if c not in ("archivo", "hoja", "fila", "col", "etiqueta")]
len(X), len(COLS)

(256335, 21)

In [3]:
archivos = sorted(X.archivo.unique())
entreno, examen = archivos[:250], archivos[250:]
A, B = X[X.archivo.isin(entreno)], X[X.archivo.isin(examen)]
modelo = RandomForestClassifier(n_estimators=60, max_depth=18, min_samples_leaf=2, n_jobs=-1, random_state=0)
modelo.fit(A[COLS], A.etiqueta)
pred = modelo.predict(B[COLS])
acierto = accuracy_score(B.etiqueta, pred)
print(f"Celdas bien clasificadas en el examen: {acierto:.2%} ({len(B)} celdas de {len(examen)} Excels)")
print(classification_report(B.etiqueta, pred, digits=4))
pd.DataFrame(confusion_matrix(B.etiqueta, pred, labels=ETIQUETAS), index=ETIQUETAS, columns=ETIQUETAS)

Celdas bien clasificadas en el examen: 99.99% (36619 celdas de 50 Excels)


              precision    recall  f1-score   support

         dia     1.0000    1.0000    1.0000      2417
        hora     1.0000    1.0000    1.0000      5154
      nombre     0.9999    0.9999    0.9999     23732
        otro     0.9994    0.9991    0.9993      3395
      prueba     1.0000    1.0000    1.0000      1921

    accuracy                         0.9999     36619
   macro avg     0.9999    0.9998    0.9998     36619
weighted avg     0.9999    0.9999    0.9999     36619



,dia,hora,nombre,prueba,otro
dia,2417,0,0,0,0
hora,0,5154,0,0,0
nombre,0,0,23730,0,2
prueba,0,0,0,1921,0
otro,0,0,3,0,3392


Algunos fallos, para ver donde se equivoca:

In [4]:
fallos = B.assign(pred=pred)[B.etiqueta != pred]
valores = etiquetas.set_index(["archivo", "hoja", "fila", "col"]).valor
fallos.assign(valor=[valores.get(k) for k in zip(fallos.archivo, fallos.hoja, fallos.fila, fallos.col)])[
    ["archivo", "valor", "etiqueta", "pred"]].head(15)

,archivo,valor,etiqueta,pred
225378,horario_257.xlsx,Victor Lopez,nombre,otro
225688,horario_258.xlsx,Lucia Martin,nombre,otro
233490,horario_270.xlsx,ver WhatsApp,otro,nombre
243403,horario_281.xlsx,ver WhatsApp,otro,nombre
243571,horario_281.xlsx,Traer guantes,otro,nombre


In [5]:
assert acierto >= 0.98, "No llega al 98%: no se guarda el modelo"
(RAIZ / "modelos").mkdir(exist_ok=True)
joblib.dump({"modelo": modelo, "columnas": COLS, "acierto_examen": acierto}, RAIZ / "modelos" / "modelo_celdas.joblib", compress=3)
print(f"Guardado: {(RAIZ / 'modelos' / 'modelo_celdas.joblib').stat().st_size / 1e6:.1f} MB")

Guardado: 0.7 MB


## Examen duro (B2b): formatos que el modelo no ha visto
Cada Excel viene de una de 30 familias (estilo de club, `familias.csv`). Dejamos fuera una familia cada vez: se entrena con las otras 29 y se examina con la que falta.
Ojo: las familias combinan las mismas opciones del generador (formato de dia, de hora, numeracion...), asi que una familia no vista es una combinacion nueva, no un formato nuevo. El examen de verdad son los Excels hechos a mano.

In [6]:
from sklearn.base import clone
X = X.merge(pd.read_csv(GEN / "familias.csv"), on="archivo", validate="many_to_one")
informe = []
for familia in sorted(X.familia.unique()):
    A, B = X[X.familia != familia], X[X.familia == familia]
    pred = clone(modelo).fit(A[COLS], A.etiqueta).predict(B[COLS])
    fallos = B[B.etiqueta != pred].assign(pred=pred[B.etiqueta != pred])
    informe.append({"familia": familia, "excels": B.archivo.nunique(), "celdas": len(B),
                    "acierto": (B.etiqueta == pred).mean(), "fallos": len(fallos),
                    "confunde": ", ".join(f"{a}->{b} ({n})" for (a, b), n in fallos.groupby(["etiqueta", "pred"]).size().items())})
informe = pd.DataFrame(informe).sort_values("acierto")
print(f"Acierto medio en familias no vistas: {informe.acierto.mean():.2%} (peor: {informe.acierto.min():.2%})")
informe.style.format({"acierto": "{:.2%}"})

Acierto medio en familias no vistas: 99.74% (peor: 96.56%)


,familia,excels,celdas,acierto,fallos,confunde
9,9,3,436,96.56%,15,"dia->nombre (13), dia->otro (2)"
19,19,10,9440,97.51%,235,"otro->nombre (31), prueba->nombre (204)"
8,8,12,8154,99.39%,50,"otro->nombre (15), prueba->nombre (35)"
28,28,7,1219,99.67%,4,nombre->otro (4)
13,13,10,4206,99.71%,12,"otro->nombre (1), prueba->nombre (11)"
16,16,6,725,99.72%,2,nombre->otro (2)
29,29,15,3020,99.83%,5,nombre->otro (5)
1,1,10,4755,99.94%,3,otro->nombre (3)
6,6,10,6498,99.94%,4,"nombre->otro (2), otro->nombre (2)"
10,10,8,1738,99.94%,1,nombre->otro (1)


## Examen con Excels hechos a mano (B2b-2)
Tres Excels inventados en `datos/a_mano/`, hechos fuera del generador: bloques por dia, tabla semanal (varios nombres por celda) y lista de una fila por reserva.
Nunca se usan para entrenar: solo examen. `etiquetas.csv` es la etiqueta de cada celda (cuadra con las reservas de `esperado.csv`).
Ojo: se mide por celda. En la tabla semanal una celda con varios nombres cuenta como una; separarlos es trabajo del lector (B3).

In [7]:
MANO = RAIZ / "datos" / "a_mano"
et_mano = pd.read_csv(MANO / "etiquetas.csv")
filas = []
for archivo in sorted(et_mano.archivo.unique()):
    for ws in load_workbook(MANO / archivo, data_only=True).worksheets:
        filas += [{"archivo": archivo, "hoja": ws.title, "fila": f, "col": c, **r} for f, c, _, r in rasgos_hoja(ws)]
M = pd.DataFrame(filas).merge(et_mano, on=["archivo", "hoja", "fila", "col"], validate="one_to_one")
M["pred"] = modelo.predict(M[COLS])
M["ok"] = M.pred == M.etiqueta
print(f"Acierto en Excels a mano: {M.ok.mean():.2%} ({len(M)} celdas)")
display(M.groupby("archivo").ok.agg(acierto="mean", bien="sum", celdas="count").style.format({"acierto": "{:.2%}"}))
M.loc[~M.ok, ["archivo", "valor", "etiqueta", "pred"]]

Acierto en Excels a mano: 100.00% (249 celdas)


,acierto,bien,celdas
archivo,,,
1_estilo_bloques.xlsx,100.00%,72,72
2_semana_tabla.xlsx,100.00%,29,29
3_lista_simple.xlsx,100.00%,148,148


,archivo,valor,etiqueta,pred
